In [ ]:
rss_category_feeds ={
    "Sports":{
        "Football":[ #Done
                "https://feeds.bbci.co.uk/sport/football/rss.xml","https://www.espn.com/espn/rss/soccer/news",
                "https://www.goal.com/feeds/en/news","https://www.skysports.com/rss/12040"
        ],
        "Cricket":[  #Done
            "https://www.espncricinfo.com/rss/content/story/feeds/0.xml","https://feeds.bbci.co.uk/sport/cricket/rss.xml","https://www.cricbuzz.com/cricket-rss-feeds",
            "https://livemint.com/rss/sports"
        ],
        "BasketBall":[  #Done
            "https://www.espn.com/espn/rss/nba/news","https://www.cbssports.com/rss/headlines/nba"
        ],
        "Hockey":[ #Done
            "https://www.espn.com/espn/rss/nhl/news","https://www.cbssports.com/rss/headlines/nhl","https://www.nbcsports.com/nhl",
            "https://www.nhl.com/rss/news.xml"
        ]
    },
    "Tech":{
        "AI":[  #Done
            "https://www.technologyreview.com/feed/","https://venturebeat.com/category/ai/feed/",
            "https://www.techrepublic.com/rssfeeds/articles","https://spectrum.ieee.org/feeds/feed.rss"
        ],
        "Startups & Business":[ #Done
            "https://techcrunch.com/category/startups/feed/","https://venturebeat.com/feed/","https://www.geekwire.com/feed/",
            "https://www.producthunt.com/feed"
        ],
        "Mobile & Apps" :[ #Done
            "https://www.androidauthority.com/feed/","https://9to5google.com/feed/","https://9to5mac.com/feed/"
        ],
        "Cybersecurity":[  #Done
            "https://krebsonsecurity.com/feed/","https://feeds.feedburner.com/TheHackersNews","https://www.darkreading.com/rss.xml"
        ]
    },
    "Finance":{
        "Stocks & Investing":[  "https://seekingalpha.com/feed.xml","https://seekingalpha.com/feed.xml","https://finance.yahoo.com/news/rssindex" #Done
        ],
        "Crypto & Blockchain":[ #Done
            "https://www.coindesk.com/arc/outboundfeeds/rss/","https://bitcoinmagazine.com/feed",
            "https://cointelegraph.com/rss","https://decrypt.co/feed"
        ],
        "India-Focused":[ #Done
            "https://www.moneycontrol.com/rss/latestnews.xml","https://economictimes.indiatimes.com/markets/rssfeeds/1977021501.cms",
            "https://www.livemint.com/rss/markets"
        ],
        "General Finance & Markets":[ #Done
            "https://feeds.bloomberg.com/markets/news.rss",
            "https://feeds.a.dj.com/rss/RSSMarketsMain.xml","https://www.economist.com/finance-and-economics/rss.xml"
        ],
        "Central Banks & Policy":[
            "https://www.federalreserve.gov/feeds/press_all.xml","https://www.imf.org/en/News/rss?language=eng"
        ]
    },
    "Science":{
        "Biology & Medicine":[
            "https://www.science.org/action/showFeed?type=axatoc&feed=rss&jc=science","https://www.nature.com/nature.rss"
        ],
        "Space & Astronomy":[  #Done
            "https://www.nasa.gov/rss/dyn/breaking_news.rss","https://www.space.com/feeds/all",
            "https://skyandtelescope.org/feed/"
        ],
        "Physics & Chemistry":[  #not working
            "https://spectrum.ieee.org/feeds/feed.rss","https://physicsworld.com/feed/"

        ],
        "General Science News":[ #not working
            "https://www.sciencedaily.com/rss/all.xml","https://www.livescience.com/feeds.xml",
            "https://phys.org/rss-feed/","https://www.scientificamerican.com/platform/syndication/rss"
        ]
    },
    "Policy":{
            "Indian Goverment":[
                "https://www.livemint.com/rss/politics","https://theprint.in/category/politics/feed/","https://www.scientificamerican.com/platform/syndication/rss/"
            ],
            "USA":[
                    "https://www.federalreserve.gov/feeds/press_all.xml","https://www.govinfo.gov/rss/usreports.xml","https://www.govinfo.gov/rss/uscourts-nyeb.xml",
                    "https://www.govinfo.gov/rss/budget.xml","https://www.govinfo.gov/rss/erp.xml"
            ],
            "International Policy Bodies":[
                "https://www.who.int/rss-feeds/news-english.xml","https://news.un.org/feed/subscribe/en/news/all/rss.xml",
                "https://news.un.org/feed/subscribe/en/news/all/rss.xml"

            ]
    }

}

In [1]:
import feedparser
import requests
import urllib.parse
from dotenv import load_dotenv
import os
from langchain_mistralai import ChatMistralAI
from langchain_core.prompts import PromptTemplate
from firecrawl import Firecrawl
import time
from langchain_core.output_parsers import PydanticOutputParser
from pydantic import BaseModel, HttpUrl, Field
from typing import List
from langchain_core.runnables import RunnableLambda, RunnableParallel
from langgraph.graph import StateGraph,START,END
from typing import TypedDict

In [2]:
load_dotenv()

True

In [ ]:
finances_url=rss_category_feeds['Policy']['International Policy Bodies']

In [ ]:
finances_url

In [ ]:
parsed_output=feedparser.parse(finances_url[0])

In [ ]:
len(parsed_output)

In [ ]:
def feedparsing(url: str) -> list:
    source = url.split(".")[1]
    result = feedparser.parse(url)
    top4_results = result["entries"][0:4]

    top4_data = []
    for res in top4_results:
        title = res["title"]
        href_url = res["links"][0]["href"]
        top4_data.append({
            "title": title,
            "link": href_url
        })

    return [{"source": source, "information": top4_data}]

In [ ]:
top4_data=feedparsing(finances_url[0])

In [ ]:
top4_data

In [ ]:
def create_runnable_football(urls: list):
    src_url = []
    for url in urls:
        source = url.split(".")[1]
        src_url.append({"source": source, "url": url})

    parallel_map = {
        item["source"]: RunnableLambda(lambda x, u=item["url"]: feedparsing(u))
        for item in src_url
    }

    parallel_chain = RunnableParallel(parallel_map) 

    return parallel_chain.invoke({})

In [ ]:
res=create_runnable_football(finances_url)

In [ ]:
res

In [ ]:
class NewsItem(BaseModel):
    """Represents a single filtered news item returned by the agent."""
    title: str = Field(..., description="The attention-grabbing news headline")
    source: str = Field(..., description="The news source (e.g. bbci, espn, skysports)")
    url: str = Field(..., description="The full URL to the article")


class FilteredNewsResponse(BaseModel):
    """Top 4 filtered news items selected by the agent."""
    items: List[NewsItem] = Field(...,min_length=1,max_length=4,description="Exactly 4 most interesting news items"
    )

In [ ]:
parser=PydanticOutputParser(pydantic_object=FilteredNewsResponse)

In [ ]:
def filter_content_finance(config) -> FilteredNewsResponse:
    model=ChatMistralAI()
    FILTER_PROMPT = """
    You are a financial news curator. Your job is to select the TOP 4 (if possible) most attention-grabbing and engaging news titles from the list below.

    ## Selection Criteria (in order of priority):
    1. **Market Moving Events** - Rate decisions, regulatory actions, major crashes or rallies, circuit breakers
    2. **Big Name Involvement** - SEBI, RBI, NSE/BSE, Fed, SEC, major indices (Nifty, Sensex, S&P 500), top stocks (Reliance, TCS, HDFC, Apple, Nvidia etc.), or influential figures (Warren Buffett, etc.)
    3. **Urgency or Breaking News** - Earnings surprises, IPO alerts, fraud/scam revelations, sudden leadership changes
    4. **Crypto & Blockchain Developments** - Exchange collapses, regulatory bans/approvals, major coin movements, hacks
    5. **Emotional or Retail Hook** - Stories that directly impact common investors, savings, or personal finance (EMI changes, inflation, tax rules)

    ## Input News List:
    {news_data}

    ## Instructions:
    - Analyze ALL titles across ALL sources
    - Pick exactly TOP 4 that will grab the most user attention
    - Cover a mix of categories if possible (e.g. not all 4 from crypto alone)
    - Avoid duplicates (same story from different sources = pick only one)
    - You MUST always include at least 1 India-focused story if present in the list
    - You MUST always pick at least 1 story even if none seem particularly interesting — choose the most relevant one available
    - No explanation, no markdown.
    {format_instructions}
    """

    parser=PydanticOutputParser(pydantic_object=FilteredNewsResponse)

    final_template=PromptTemplate(
       template=FILTER_PROMPT,
       input_variables=["news_data"],
       partial_variables={"format_instructions":parser.get_format_instructions()}
    )

    chain = final_template | model | parser 

    result=chain.invoke({"news_data":config})
    return result

In [ ]:
def filter_content_science(config) -> FilteredNewsResponse:
    model=ChatMistralAI()
    FILTER_PROMPT = """
    You are a science news curator. Your job is to select the TOP 4 (if possible) most attention-grabbing and engaging news titles from the list below.

    ## Selection Criteria (in order of priority):
    1. **Breakthrough Discoveries** - New cures, vaccines, space discoveries, particle physics findings, climate tipping points
    2. **Big Institution / Mission / Name Involvement** -
        - Biology & Medicine: WHO, NIH, ICMR, FDA, top journals (Nature, Science, Lancet), landmark studies, known researchers
        - Space & Astronomy: NASA, ISRO, ESA, SpaceX, James Webb Telescope, Mars/Moon missions, black holes, exoplanets
        - Physics & Chemistry: CERN, Nobel Prize winners, quantum computing breakthroughs, new elements or materials
        - Environment & Climate: IPCC, UNEP, COP summits, extreme weather events, endangered species, deforestation alerts
    3. **Urgency or Public Impact** - Disease outbreaks, asteroid threats, pollution crises, drug approvals or bans, radiation incidents
    4. **Human or Emotional Hook** - First-ever achievements, stories affecting everyday life, health warnings for common people, extinction news
    5. **India-Focused Developments** - ISRO missions, Indian research breakthroughs, local environmental crises, ICMR findings, Indian scientists

    ## Input News List:
    {news_data}

    ## Instructions:
    - Analyze ALL titles across ALL sources
    - Pick exactly TOP 4 that will grab the most user attention
    - Cover a mix of science categories if possible (e.g. not all 4 from space alone)
    - Avoid duplicates (same story from different sources = pick only one)
    - You MUST always include at least 1 India-focused story (ISRO, Indian research, local environment) if present in the list
    - You MUST always pick at least 1 story even if none seem particularly interesting — choose the most relevant one available
    - No explanation, no markdown.
    {format_instructions}
    """

    parser=PydanticOutputParser(pydantic_object=FilteredNewsResponse)

    final_template=PromptTemplate(
       template=FILTER_PROMPT,
       input_variables=["news_data"],
       partial_variables={"format_instructions":parser.get_format_instructions()}
    )

    chain = final_template | model | parser 

    result=chain.invoke({"news_data":config})
    return result

In [ ]:
def filter_content_policy(config) -> FilteredNewsResponse:
    model=ChatMistralAI()
    FILTER_PROMPT = """
    You are a policy and governance news curator. Your job is to select the TOP 4 (if possible) most attention-grabbing and engaging news titles from the list below.

    ## Selection Criteria (in order of priority):
    1. **Controversy or Political Drama** - Scandals, resignations, no-confidence motions, diplomatic fallouts, protests, 
        election controversies, impeachments, policy reversals
    2. **Big Institution / Leader / Body Involvement** -
        - India Policy: PMO, Parliament, Supreme Court, Election Commission, NITI Aayog, RBI, CBI, ED, 
          key ministers (PM Modi, Home Minister, Finance Minister), state elections, landmark bills & amendments
        - USA Policy: White House, Congress, Supreme Court, FBI, CIA, Federal Reserve, President, key senators,
          executive orders, major legislation (budget, healthcare, immigration)
        - International Policy Bodies: United Nations, WHO, WTO, IMF, World Bank, NATO, G20, G7, BRICS, 
          ICC, ASEAN, EU Parliament, major treaties & sanctions
    3. **Urgency or Breaking News** - Election results, emergency declarations, sudden policy reversals,
        surprise appointments or resignations, war declarations, ceasefire announcements
    4. **Cross-Border or Geopolitical Impact** - India-Pakistan, India-China, US-China, Russia-Ukraine, 
        Middle East conflicts, trade wars, sanctions, border disputes, terrorism-related policy
    5. **Citizen or Public Impact Hook** - Policies directly affecting common people — tax changes, 
        reservation policies, subsidies, welfare schemes, visa rule changes, internet shutdowns

    ## Input News List:
    {news_data}

    ## Instructions:
    - Analyze ALL titles across ALL sources
    - Pick exactly TOP 4 that will grab the most user attention
    - Cover a mix of policy categories if possible (e.g. not all 4 from India alone)
    - Avoid duplicates (same story from different sources = pick only one)
    - You MUST always include at least 1 India-focused story (Indian Parliament, Supreme Court, state politics) if present in the list
    - You MUST always prioritize geopolitically sensitive or high-tension stories over routine policy updates
    - You MUST always pick at least 1 story even if none seem particularly interesting — choose the most relevant one available
    - No explanation, no markdown.
    {format_instructions}
    """

    parser=PydanticOutputParser(pydantic_object=FilteredNewsResponse)

    final_template=PromptTemplate(
       template=FILTER_PROMPT,
       input_variables=["news_data"],
       partial_variables={"format_instructions":parser.get_format_instructions()}
    )

    chain = final_template | model | parser 

    result=chain.invoke({"news_data":config})
    return result

In [ ]:
fil_res=filter_content_policy(res)

In [ ]:
fil_res.items

In [ ]:
fil_sci=filter_content_science(res)

In [ ]:
fil_sci.items